In [5]:
import openmeteo_requests
import polars as pl
import requests_cache
import os
from retry_requests import retry
from datetime import datetime, timedelta, timezone

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


# Get location information
locations_df = pl.read_csv("meta_data_used_stations.csv")

location_Zst = locations_df["Zst"].to_list()
location_names = locations_df["name"].to_list()
latitudes = locations_df["lat"].to_list()
longitudes = locations_df["lon"].to_list()

# URL and parameters remain the same
url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
    "latitude": latitudes,
    "longitude": longitudes,
    "start_date": "2021-01-01",
    "end_date": "2025-12-31",
    "hourly": ["european_aqi", "pm10", "pm2_5", "nitrogen_dioxide", "carbon_monoxide", "ozone", "sulphur_dioxide", "carbon_dioxide"]
}
responses = openmeteo.weather_api(url, params=params)
print("finished")


finished


In [4]:
# Hourly Data
BASE_DIR   = "../../data"
OUTPUT_DIR = os.path.join(BASE_DIR, "AirQuality", "air_quality_hourly.csv")


all_dataframes = []
# Process first location
for i, response in enumerate(responses):
    # Process location
    response = responses[0]
    print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

    # Get name
    zst = location_Zst[i]
    name = location_names[i]
    
    # --- Process Hourly Data ---
    hourly = response.Hourly()
    
    #  calculate the range using datetime objects
    start_h = datetime.fromtimestamp(hourly.Time(), tz=timezone.utc)
    end_h = datetime.fromtimestamp(hourly.TimeEnd(), tz=timezone.utc)
    
    interval_h = timedelta(seconds=hourly.Interval())
    
    hourly_dataframe = pl.DataFrame({
        "date": pl.datetime_range(start_h, end_h - interval_h, interval_h, eager=True),
        "location_Zst": zst,
        "location_name": name,
        "european_aqi": hourly.Variables(0).ValuesAsNumpy(),
        "pm10": hourly.Variables(1).ValuesAsNumpy(),
        "pm2_5": hourly.Variables(2).ValuesAsNumpy(),
        "nitrogen_dioxide": hourly.Variables(3).ValuesAsNumpy(),
        "carbon_monoxide": hourly.Variables(4).ValuesAsNumpy(),
        "ozone": hourly.Variables(5).ValuesAsNumpy(),
        "sulphur_dioxide": hourly.Variables(6).ValuesAsNumpy(),
        "carbon_dioxide": hourly.Variables(7).ValuesAsNumpy(),
    })
    
    all_dataframes.append(hourly_dataframe)

final_dataframe = pl.concat(all_dataframes)

final_dataframe.write_csv(
    OUTPUT_DIR,
    datetime_format="%d.%m.%Y %H:%M"
)

print("\nHourly data\n", hourly_dataframe)


Coordinates: 54.400001525878906°N 10.0°E
Elevation: 20.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 54.400001525878906°N 10.0°E
Elevation: 20.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 54.400001525878906°N 10.0°E
Elevation: 20.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 54.400001525878906°N 10.0°E
Elevation: 20.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 54.400001525878906°N 10.0°E
Elevation: 20.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 54.400001525878906°N 10.0°E
Elevation: 20.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 54.400001525878906°N 10.0°E
Elevation: 20.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 54.400001525878906°N 10.0°E
Elevation: 20.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
 shape: (43_824, 11)
┌────────────┬────────────┬────────────┬───────────┬───┬───────────┬───────┬───────────┬───────────┐
│ date       ┆ location_Z ┆ location_n ┆ european_ ┆ … ┆ carbon_mo ┆ ozone ┆ sulphur_d ┆ carbon_d